# Initial EDA

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from schneider_forecast.config import (
    BOM_FILES,
    BOM_REFERENCE,
    DATA_RAW,
    ENERGY_PRICES_WIDE,
    FIGURES,
    HORIZONS,
    QUANTILES,
    RM_PRICES_WIDE,
)

FIG_DIR = FIGURES / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.style.use("tableau-colorblind10")

# 2021-22 commodity surge, shaded on time-series plots
SURGE = ("2021-01-01", "2022-12-31")


def save_fig(fig: plt.Figure, name: str) -> None:
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## 1. Data inventory

### 1.1 Raw Material prices

In [ ]:
def series_inventory(df: pd.DataFrame) -> pd.DataFrame:
    """First and last observation, count and inner gaps of each column."""
    rows = []
    for col in df.columns:
        s = df[col]
        obs = s.dropna()
        rows.append(
            {
                "series_id": col,
                "first": obs.index.min(),  # first date
                "last": obs.index.max(),  # last date
                "n_obs": obs.size,  # number of non-NaN observations
                # NaNs between first and last observation, i.e. real gaps, not just late start/end
                "gaps_inside": s.loc[obs.index.min() : obs.index.max()].isna().sum()
                if obs.size
                else pd.NA,
            }
        )
    return pd.DataFrame(rows).set_index("series_id")

In [ ]:
rm = pd.read_csv(RM_PRICES_WIDE, parse_dates=["month"], index_col="month").drop(
    columns=["year", "month_num"]
)
rm_catalog = pd.read_csv(DATA_RAW / "csv" / "source_catalog.csv", index_col="series_id")

# catalog columns added so each series ID reads with its material, region and unit
rm_inventory = series_inventory(rm).join(rm_catalog[["material", "region", "unit", "proxy_type"]])
rm_inventory

S, D series: no NaNs
L series: empty

In [ ]:
# licensed L series are entirely empty, so they are dropped from the analysis
rm_obs = rm.dropna(axis=1, how="all")
series_label = (rm_catalog["material"] + " · " + rm_catalog["region"]).to_dict()
rm_obs.columns.tolist()

### 1.2 Energy

In [ ]:
energy = pd.read_csv(ENERGY_PRICES_WIDE, parse_dates=["month"], index_col="month")
energy_catalog = pd.read_csv(DATA_RAW / "csv" / "energy_source_catalog.csv", index_col="series_id")
series_inventory(energy).join(energy_catalog[["energy", "region", "unit", "relevance"]])

no NaNs, all have about 119 month observations

### 1.3 BOM

In [ ]:
# reference BOM = the latest version Schneider sent (20 Nov v1)
bom_raw = pd.read_excel(BOM_FILES[BOM_REFERENCE], sheet_name="BOM_Materials")
# the last rows of the file are mass totals, not materials; real rows have a plant code.
# plant names are dropped: only the plant code and country matter for pricing
bom = bom_raw[bom_raw["plant_code"].notna()].drop(columns="plant_name").copy()
bom

In [ ]:
bom_products = bom["product_ref"].unique()
bom_products

[CLAUDE to VALIDATE]
2 products, RMAirSet and RM6
share_pct is the percent of the total BOM
series_ids are the equivalent RM series prices. 

### 1.4 Steel prices (MEPS)

In [ ]:
meps_raw = pd.read_excel(
    DATA_RAW / "Copia de Europe Steel INDEXES.xlsx", sheet_name="MEPS", header=None
)
# one 5-column block per country: the country name sits in row 0 above the block,
# product names (HR / CR / HDG / EG coil) in row 4, monthly €/t values from row 5
country = meps_raw.iloc[0].where(meps_raw.iloc[0] != "x").ffill()
product = meps_raw.iloc[4]
keep = [
    c
    for c in meps_raw.columns[1:]
    if country[c] in ("Germany", "France") and str(product[c]).endswith("coil")
]
steel = meps_raw.iloc[5:, [0, *keep]].set_axis(
    ["month", *[f"{country[c]} {product[c]}" for c in keep]], axis=1
)
steel["month"] = pd.to_datetime(steel["month"], errors="coerce")
# date rows run past the last value, so fully empty months are dropped
steel = (
    steel.dropna(subset=["month"])
    .set_index("month")
    .apply(pd.to_numeric, errors="coerce")
    .dropna(how="all")
)
series_inventory(steel)

168 months of observation - earlier time window

### Stainless steel prices (CRU)

In [ ]:
cru_ss = pd.read_excel(
    DATA_RAW / "Copia de Europe Steel INDEXES.xlsx", sheet_name="CRU Stainless", header=None
)
# country (row 0) and grade (row 1) sit above the first column of each block; each grade has
# two columns from row 5: BASE price and alloy surcharge, in €/t, dated on the 15th
ss_country = cru_ss.iloc[0].ffill()
ss_grade = cru_ss.iloc[1].ffill()
cols_304 = [
    c for c in cru_ss.columns[1:] if ss_country[c] == "Germany" and ss_grade[c] == "Grade 304"
]
ss = cru_ss.iloc[5:, [0, *cols_304]].set_axis(["month", "base", "alloy"], axis=1)
ss["month"] = pd.to_datetime(ss["month"], errors="coerce")
ss = ss.dropna(subset=["month"]).set_index("month").apply(pd.to_numeric, errors="coerce")
ss.index = ss.index.to_period("M").to_timestamp()
# price paid = base price + alloy surcharge (the surcharge passes nickel and chrome costs on)
stainless = (ss["base"] + ss["alloy"]).dropna().rename("Germany stainless 304")
series_inventory(stainless.to_frame())

### 1.5 Epoxy and LME

In [ ]:
epoxy = pd.read_excel(DATA_RAW / "Epoxy for student.xlsx", index_col=0).dropna(axis=1, how="all")
epoxy.columns = ["benzene", "acetone", "epichlorohydrin", "other_costs", "epoxy_resin"]
# a few dates are not the 1st of the month (e.g. 2026-09-03): snap them to month start
epoxy.index = pd.to_datetime(epoxy.index).to_period("M").to_timestamp()

lme_raw = pd.read_excel(
    DATA_RAW / "LME_Copper_Aluminium_EU_Monthly_2016_2026.xlsx",
    sheet_name="Monthly Prices",
    header=4,
)
# the last month is month-to-date (provisional): keep completed months only
lme = lme_raw[lme_raw["Period status"] == "Completed month"].set_index("Month")

pd.concat([series_inventory(epoxy), series_inventory(lme.select_dtypes("number"))])

### 1.6 RMU price benchmark

In [ ]:
rmu = pd.read_excel(
    DATA_RAW / "RMU_IQI_EU_Price_Benchmark_2017-2026_Synthetic.xlsx",
    sheet_name="ML_Trend",
    header=3,
)
rmu = rmu.rename(columns={"P10 (ML)": "P10", "P50 (ML)": "P50", "P90 (ML)": "P90"})[
    ["ID", "Brand", "Year", "P10", "P50", "P90"]
]
# one row per year, one column per brand
rmu_p50 = rmu.pivot(index="Year", columns="Brand", values="P50")
rmu_p50

## 2. Behaviour over time

In [ ]:
def plot_rebased(df: pd.DataFrame, title: str, name: str, labels: dict | None = None) -> None:
    # each series divided by its first observed value: all start at 100, so a USD/t
    # price and an index can share one axis and only relative moves are compared
    rebased = df.div(df.apply(lambda s: s.dropna().iloc[0])) * 100
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for col in rebased.columns:
        ax.plot(rebased.index, rebased[col], label=(labels or {}).get(col, col))
    ax.axvspan(*pd.to_datetime(SURGE), color="grey", alpha=0.15, label="2021–22 surge")
    ax.set(title=title, xlabel="Month", ylabel="Index, first month = 100")
    ax.legend(fontsize=8, ncol=2)
    save_fig(fig, name)
    plt.show()


plot_rebased(
    rm_obs[["S01", "S02", "S03", "S04", "S05"]],
    "LME metals and iron ore",
    "trend_metals",
    series_label,
)
plot_rebased(
    rm_obs[["S07", "S08", "S09", "S10", "S11"]],
    "EU producer price indices",
    "trend_eu_ppi",
    series_label,
)
# Germany only: French prices track it closely
plot_rebased(
    steel.loc["2016-11":, steel.columns.str.startswith("Germany")],
    "Steel prices, MEPS Germany (€/t)",
    "trend_steel_meps",
)

spikes observed between 2021-2023, may affect future forecasts
decision point: how to treat this window of data given uneven surges but also the fact that it spikes the historical 

## 3. Movement over 3/6/9 months

In [ ]:
# RM series, MEPS steel and CRU stainless on the same months (Nov 2016 onwards)
prices = rm_obs.join(steel).join(stainless)
series_label.update({c: c for c in prices.columns if c not in series_label})

# For each start month t and horizon h:  change_h(t) = price(t + h) / price(t) - 1
# shift(-h) moves the price h months later onto row t, so each row holds the move that
# starts in that month (~120 - h moves per series). Consecutive windows overlap by h - 1
# months, so the moves are not independent: the effective sample is smaller than it looks.
changes = {h: prices.shift(-h) / prices - 1 for h in HORIZONS}

# P10 / P50 / P90 of those moves: 80% of historical h-month moves fall between P10 and
# P90 (a first guess at forecast band width), P50 is the typical drift. % changes have no
# unit, so USD/t prices and indices sit in the same table.
volatility = pd.concat(
    {
        h: c.quantile(list(QUANTILES)).T.set_axis(["P10", "P50", "P90"], axis=1)
        for h, c in changes.items()
    },
    axis=1,
)
volatility.columns.names = ["horizon (months)", "percentile"]
(volatility * 100).round(1)

In [ ]:
# 6-month moves per series, sorted by the width of the P10–P90 band
band_6m = (volatility[(6, "P90")] - volatility[(6, "P10")]).sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(
    [changes[6][col].dropna() * 100 for col in band_6m.index],
    tick_labels=[series_label.get(c, c) for c in band_6m.index],
    vert=False,
    whis=(10, 90),
    showfliers=False,
)
ax.axvline(0, color="grey", lw=0.8)
ax.set(
    title="6-month price moves per series (whiskers = P10 and P90)",
    xlabel="% change over 6 months",
    ylabel="Series",
)
save_fig(fig, "moves_6m_boxplot")
plt.show()

1. Largest volatility within 6 months is on top, so Iron ore, nickel. 
2. Most prices with large range of P10 and P90 also have large quartile ranges
3. Percent changes all look to be under 5% plus minus in 6 months

## 4. Predictability

In [ ]:
# Naive forecast: price in h months = price today. Its error is measured as MAPE
# (mean absolute percentage error): |actual - forecast| / actual, averaged over forecast
# origins from 2020 onwards. This is the bar every model has to beat.
naive_mape = pd.DataFrame(
    {
        h: ((prices.shift(-h) - prices).abs() / prices.shift(-h)).loc["2020":].mean()
        for h in HORIZONS
    }
)
naive_mape.columns.name = "horizon (months)"
(naive_mape * 100).round(1).sort_values(6)

nevermind, with the exception of S04 and S03, steel prices have the largest increases over 6 months. S04 and S03 are Iron ore and nickel

In [ ]:
# Persistence: correlation between this month's change and last month's change.
# Positive = moves tend to continue (momentum), negative = they tend to reverse.
monthly_change = prices.pct_change(fill_method=None)
persistence = monthly_change.apply(lambda s: s.autocorr(lag=1)).rename("lag-1 autocorrelation")
persistence.round(2).sort_values()

persistence: not much

## 5. Co-movement and leading indicators

In [ ]:
# correlation of 3-month changes, not levels: trending levels look correlated even when
# the series are unrelated
corr = prices.pct_change(3, fill_method=None).corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
fig.colorbar(im, ax=ax, label="Correlation of 3-month % changes")
ax.set_title("Co-movement of price series (3-month changes)")
save_fig(fig, "corr_3m_changes")
plt.show()

large correlation with steel prices basically 1, also some S values and D values. 

In [ ]:
def lead_lag(driver: pd.Series, target: pd.Series, lags=(0, 3, 6, 9)) -> pd.Series:
    # correlation of the driver's 3-month change at month t with the target's 3-month
    # change k months later: shift(-k) brings the target's future change onto row t
    d = driver.pct_change(3, fill_method=None)
    t = target.pct_change(3, fill_method=None)
    return pd.Series({k: d.corr(t.shift(-k)) for k in lags})


pairs = {
    "EU gas (E05) → aluminium (S02)": (energy["E05"], prices["S02"]),
    "German power (E09) → EU steel PPI (S09)": (energy["E09"], prices["S09"]),
    "German power (E09) → CR coil DE": (energy["E09"], prices["Germany CR coil"]),
    "Oil (E01) → EU plastics PPI (S11)": (energy["E01"], prices["S11"]),
}
lead_lag_table = pd.DataFrame({name: lead_lag(d, t) for name, (d, t) in pairs.items()}).T
lead_lag_table.columns.name = "target k months later"
lead_lag_table.round(2)

## 6. Business weight

In [ ]:
# main price series per BOM material: a real price where one exists, else the closest proxy.
# Brass, "Other" and the unidentified remainder have no price series and are left out (unpriced);
# epoxy resin keeps the BOM's plastics mapping (S11)
main_series = {
    "Carbon steel": "Germany CR coil",
    "Stainless steel": "Germany stainless 304",
    "Copper": "D01",
    "Galvanised steel": "Germany HDG coil",
    "Aluminium": "D02",
    "Engineering plastic": "S11",
    "Plastic": "S11",
}
exposure = bom[bom["material_family"].isin(main_series)].copy()
exposure["series"] = exposure["material_family"].map(main_series)
# steel parts are priced on their plant country's index: RM6 is made in France
rm6_steel = exposure["product_ref"].str.startswith("RM6") & exposure["material_family"].eq(
    "Carbon steel"
)
exposure.loc[rm6_steel, "series"] = "France CR coil"
# the v1 BOM gives every mass, so no share needs to be assumed
exposure["mass_used"] = exposure["mass_kg"]

# € per kg in the last observed month (€/t ÷ 1000) for series quoted in €/t;
# plastics only exist as an index here, so their level is an assumption
latest_eur_kg = prices.ffill().iloc[-1] / 1000
assumed_eur_kg = {"Engineering plastic": 3.0, "Plastic": 3.0}
exposure["eur_per_kg"] = (
    exposure["material_family"].map(assumed_eur_kg).fillna(exposure["series"].map(latest_eur_kg))
)

exposure["cost_eur"] = exposure["mass_used"] * exposure["eur_per_kg"]
exposure["cost_share"] = exposure["cost_eur"] / exposure.groupby("product_ref")[
    "cost_eur"
].transform("sum")
exposure["band_6m"] = exposure["series"].map(band_6m)
# € at risk = material cost × width of the 6-month P10–P90 band: € swing per unit in 6 months
exposure["eur_at_risk"] = exposure["cost_eur"] * exposure["band_6m"]
exposure[
    [
        "product_ref",
        "material_family",
        "mass_used",
        "eur_per_kg",
        "cost_eur",
        "cost_share",
        "band_6m",
        "eur_at_risk",
    ]
].round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, layout="constrained")
for ax, (product, grp) in zip(axes, exposure.groupby("product_ref"), strict=True):
    grp = grp.groupby("material_family")["eur_at_risk"].sum().sort_values()
    ax.barh(grp.index, grp.values)
    ax.set(title=product, xlabel="6-month P10–P90 cost swing per unit (€)")
axes[0].set_ylabel("Material")
fig.suptitle("Where raw-material price risk sits in each product")
save_fig(fig, "exposure_by_material")
plt.show()

shows the highest risk raw material prices by multiplying the cost x 6m band between P10 and P90 per unit. 

## 7. Epoxy (insulating material)

In [ ]:
plot_rebased(
    epoxy[["benzene", "acetone", "epichlorohydrin", "epoxy_resin"]],
    "Epoxy resin and its inputs",
    "trend_epoxy",
)

In [ ]:
# same measures as sections 3 and 4, for the resin alone (it starts in 2018, so it is
# kept out of `prices` to leave the other series on a common window)
resin = epoxy["epoxy_resin"]
resin_moves = {h: resin.shift(-h) / resin - 1 for h in HORIZONS}
epoxy_summary = pd.DataFrame(
    {
        **{
            f"P{round(q * 100)}": {h: m.quantile(q) for h, m in resin_moves.items()}
            for q in QUANTILES
        },
        "naive MAPE (2020+)": {
            h: ((resin.shift(-h) - resin).abs() / resin.shift(-h)).loc["2020":].mean()
            for h in HORIZONS
        },
    }
)
epoxy_summary.index.name = "horizon (months)"
(epoxy_summary * 100).round(1)

In [ ]:
# do input prices move before the resin price? same method as section 5
pd.DataFrame(
    {
        f"{x} → epoxy resin": lead_lag(epoxy[x], resin)
        for x in ["benzene", "acetone", "epichlorohydrin"]
    }
).T.round(2)

## 8. RMU unit prices

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for brand in rmu_p50.columns:
    ax.plot(rmu_p50.index, rmu_p50[brand], marker="o", label=brand)
se = rmu[rmu["ID"] == "SE"].set_index("Year")
ax.fill_between(se.index, se["P10"], se["P90"], alpha=0.15, label="Schneider P10–P90")
ax.set(title="RMU unit price by brand (P50)", xlabel="Year", ylabel="Unit price (€, ex-VAT)")
ax.legend(fontsize=8, ncol=2)
save_fig(fig, "rmu_price_by_brand")
plt.show()

RMU prices show a high spike in per unit price in 2026 (Schneider +0.1 % in 2025, +40 % in 2026). 

In [ ]:
latest = rmu_p50.index.max()
yearly_change = rmu_p50.pct_change()
band_width = rmu.assign(width=(rmu["P90"] - rmu["P10"]) / rmu["P50"]).pivot(
    index="Year", columns="Brand", values="width"
)
rmu_summary = pd.DataFrame(
    {
        f"P50 {latest} (€)": rmu_p50.loc[latest],
        "vs Schneider (%)": (rmu_p50.loc[latest] / rmu_p50.loc[latest, "Schneider Electric"] - 1)
        * 100,
        f"change in {latest} (%)": yearly_change.loc[latest] * 100,
        f"avg yearly change 2018–{latest - 1} (%)": yearly_change.loc[2018 : latest - 1].mean()
        * 100,
        "P10–P90 width (% of P50)": band_width.loc[latest] * 100,
    }
)
rmu_summary.round(1).sort_values(f"P50 {latest} (€)")

In [ ]:
# Does Schneider's price follow its material cost? Fixed-weight cost index for RM6: each
# material's price divided by its latest level, weighted by its share of RM6 material cost.
# Yearly averages, to match the yearly price data (the last year is a partial year).
rm6 = exposure[exposure["product_ref"].str.startswith("RM6")]
weights = rm6.groupby("series")["cost_share"].sum()
relative = prices[weights.index] / prices[weights.index].ffill().iloc[-1]
cost_index = relative.mul(weights).sum(axis=1, min_count=len(weights))
yearly_cost = cost_index.groupby(cost_index.index.year).mean()

pass_through = pd.DataFrame(
    {
        "RM6 material cost change (%)": yearly_cost.pct_change() * 100,
        "Schneider price change (%)": rmu_p50["Schneider Electric"].pct_change() * 100,
    }
).loc[2018:latest]
# cost change one year earlier: a price that follows cost with a delay matches this column
pass_through["cost change, year before (%)"] = pass_through["RM6 material cost change (%)"].shift(1)
pass_through.round(1)

In [ ]:
# correlation of the price change with the same-year and previous-year cost change
pass_through.corr()["Schneider price change (%)"].round(2)

## 9. Material cost vs unit price

In [ ]:
# material cost per unit from section 6 against Schneider's latest P50 price. The benchmark
# does not say which configuration it prices, so both products are compared to the same price
material_cost = exposure.groupby("product_ref")[["cost_eur", "mass_used"]].sum()
se_price = rmu_p50.loc[latest, "Schneider Electric"]
pd.DataFrame(
    {
        "BOM mass priced (kg)": material_cost["mass_used"],
        "BOM material cost (€)": material_cost["cost_eur"],
        f"Schneider P50 price {latest} (€)": se_price,
        "material cost / price (%)": material_cost["cost_eur"] / se_price * 100,
    }
).round(1)

## 10. Last observed month per series

In [ ]:
# last observed month per series = where its forecast starts. Publication lag = months after
# the month ends before the value is released (assumed: EU PPIs 2, US PPIs 1, prices 0);
# a feature may only use values older than its lag
all_series = pd.concat([rm_obs, energy, steel, stainless, epoxy], axis=1)
pub_lag = {
    **dict.fromkeys(["S07", "S08", "S09", "S10", "S11"], 2),
    **dict.fromkeys(["S12", "S13"], 1),
    **dict.fromkeys([f"E{i}" for i in range(13, 20)], 2),
}
coverage = series_inventory(all_series)[["first", "last", "n_obs"]]
coverage["lag_months (assumed)"] = coverage.index.to_series().map(pub_lag).fillna(0).astype(int)
coverage.sort_values(["last", "lag_months (assumed)"])

## 11. Findings

- Prices were flat in 2017–20, surged in 2021–22, then partly unwound. The surge dominates volatility: decide how to treat it.
- Big rises go further than big falls, and the gap widens with the horizon. 9-month P10 to P90: copper −12% to +36%, steel −17% to +30%.
- No-change forecast error at 6 months: 6–9% for PPIs, 12–15% for LME metals, 15–19% for steel coils.
- PPIs follow trends (month-to-month persistence 0.6–0.8). LME metals barely do (0.1–0.3).
- The 8 MEPS coil series move as one. Iron ore and nickel are weak proxies for EU carbon steel and stainless.
- Energy leads: German electricity leads the EU steel PPI by about 6 months (0.41), oil leads plastics by 0–3 months (0.45).
- Cost risk per unit (6-month € swing, v1 BOM): RM6 copper €154, stainless €70, carbon steel €39; RM AirSeT copper €113, carbon steel €99, stainless €94. Real stainless 304 is the riskiest material (6-month naive error 16%, 9-month band −22% to +43%).
- Data end dates: stainless ends 2026-08 (needs a 1-month nowcast for the 2026-09 origin); S06, MEPS steel and some energy run to 2026-10 and must be cut at 2026-09.
- Epoxy: steady and easy to forecast (6-month P10 to P90 −10% to +15.5%, naive error 8%). Acetone leads the resin by ~3 months (0.41); epichlorohydrin moves the same month (0.65).
- RMU prices: Schneider has the tightest band (P10–P90 = 28% of P50). Eaton, the only SF6-free competitor, rose +8% in 2026 vs +32–40% for others and is now 5.5% cheaper than Schneider.
- Price vs cost: price does not follow material cost one-for-one. 2021: RM6 cost +58%, price +6.7% (absorbed); 2026: cost +21%, price +40%. Correlation 0.38 same year, 0.00 with a 1-year delay (~8 points).
- Material cost vs price (v1 BOM): priced raw material is €900 (RM6) and €975 (RM AirSeT), 7.4% and 8.0% of the 2026 unit price (€12,233). AirSeT now costs more than RM6 in raw materials because the v1 BOM adds 28.5 kg of copper (€358).